# SentinelPay\n\nNotebook didatico para treinar e avaliar o modelo de risco de fraude da SentinelPay usando o dataset publico de cartoes de credito.

In [ ]:
!pip install -q pandas==2.2.3 numpy==1.26.4 scikit-learn==1.5.2 imbalanced-learn==0.12.4 xgboost==2.1.2 matplotlib==3.9.2

In [ ]:
import numpy as np\nimport pandas as pd\nfrom imblearn.over_sampling import SMOTE\nfrom sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom xgboost import XGBClassifier

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'\ndf = pd.read_csv(url)\ndf.shape, df['Class'].sum(), df['Class'].mean()

In [ ]:
df['Amount_log'] = np.log1p(df['Amount'])\nX = df.drop(columns=['Class', 'Amount'])\ny = df['Class'].astype(int)\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.30, stratify=y, random_state=42\n)\n\nscaler = StandardScaler()\nX_train = X_train.copy()\nX_test = X_test.copy()\nX_train['Amount_log'] = scaler.fit_transform(X_train[['Amount_log']])\nX_test['Amount_log'] = scaler.transform(X_test[['Amount_log']])

In [ ]:
smote = SMOTE(random_state=42)\nX_res, y_res = smote.fit_resample(X_train, y_train)\n\nmodel = XGBClassifier(\n    n_estimators=250,\n    learning_rate=0.05,\n    max_depth=4,\n    subsample=0.90,\n    colsample_bytree=0.90,\n    scale_pos_weight=10,\n    eval_metric='logloss',\n    random_state=42,\n    n_jobs=-1,\n)\nmodel.fit(X_res, y_res)

In [ ]:
threshold = 0.30\ny_prob = model.predict_proba(X_test)[:, 1]\ny_pred = (y_prob >= threshold).astype(int)\n\nprint(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))\nprint('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))\nprint('Average precision:', round(average_precision_score(y_test, y_prob), 4))\nconfusion_matrix(y_test, y_pred)